# Building a neural network from scratch

In this module, you will build a neural network which will learn a very simple function, a polynomial function of the sort: 
y = A + B * x + C * x^2 + D * x^3 

You can choose the value of coefficients for this exercise. Later on we will see what happens when we add noise. 

But first, let us plot and see your function! 

In [ ]:
import numpy as np 
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from src.utils import degree_3_polynomial
A = 1 
B = 2 
C = -0.5 
D = 0.04 
x_true = np.linspace(0, 10, 1000)
y_true = degree_3_polynomial(x_true, coeffs=[A, B, C, D], noise_std=0.2) # gaussian noise with stdev added

plt.plot(x_true, y_true, color="black")
plt.xlabel('X')
plt.ylabel('Y')

# Learning from data
The function that you plot above is ofcourse arbitrary, since you chose the coefficients of the polynomial. In fact the choice of a polynomial is also arbitrary (we chose it because it is easy to explain). But in reality this could have a certain meaning; a 1-D function could be a signal from a sensor, for instance a temperature sensor, or it could show how price of a house changes with time, etc. 
(Insert example of a 1-D signal relevant to biology, which is simple like a polynomial)

In reality, you will get only discrete data points from a sensor and you need to build a model from this data. The process of learning the function from a discrete set of points is called Machine Learning. Below, you can see a set of sampled points from the function that you generated above. The sampled points are in red, and the dashed line indicates the true function. 

In [ ]:
sample_size = 20
sampling_indices = np.random.choice(len(x_true), size=sample_size, replace=False)
x_sample = x_true[sampling_indices]
y_sample = y_true[sampling_indices] 

plt.plot(x_true, y_true, color="black", label="True Function", linestyle='--', alpha=0.2)
plt.scatter(x_sample, y_sample, color="red", label="Sampled Data")
plt.legend()

# show sampled points as a table 

data = {'X': x_sample, 'Y': y_sample}
df = pd.DataFrame(data)
print(df.head())

## Fit with a polynomial function

A direct method is ofcourse to find the coefficients of a polynomial function using the least squares method. We use standard NumPy libraries to do this. Click [here](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html) to see more details. 


In [ ]:
# Fitting a polynomial of degree 3 to the data
coefficients = np.polyfit(x_sample, y_sample, deg=3) # to get the coefficients of the fitted polynomial
fitted_polynomial = np.poly1d(coefficients) # to create a polynomial function from the coefficients
# find the fit function using true x values
y_fitted = fitted_polynomial(x_true)
plt.plot(x_true, y_true, color="black", label="True Function", linestyle='--', alpha=0.2) 
plt.plot(x_true, y_fitted, color="blue", label="Fitted Polynomial")
plt.scatter(x_sample, y_sample, color="red", label="Sampled Data")
plt.xlabel('X')
plt.ylabel('Y')
plt.ylim(-1, 12)
plt.legend()


# Learning through neurons

We will see in this module how the same function can be learnt through a different framework, one loosely inspired by the human brain. A neuron is simply a cell which takes some input and transforms into an output, and the comparison to biological neuron basically ends there. 

For biological neurons, the transformation of the input is generally described as a binary output (either it fires or not). This can be modelled as a sigmoid function, where the output slowly changes between 0 and 1 through an exponential change. 

A digital neuron can ofcourse be modelled by any mathematical function. Normally, this function is a non-linear function. There are different types of non-linear functions that are used to build artificial neural nets. In this exercise, we will focus on three functions: _ReLU_, _sigmoid_ and _tanh_.

The ReLU function is defined as:
\begin{equation}
f(x) = \begin{cases} 
      0 & x < 0 \\
      x & x \geq 0 
   \end{cases}
\end{equation}

The sigmoid function is defined as:
\begin{equation}
f(x) = \frac{1}{1 + e^{-x}}
\end{equation}

The tanh function is defined as:
\begin{equation}
f(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}
\end{equation}

Can you implement the ReLU, sigmoid and tanh functions in Python? 


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
def activation_function(z, function_type="sigmoid"):
    if function_type == "linear":
        return z # a linear activation function simply returns the same input
    if function_type == "sigmoid":
        return 1 / (1 + np.exp(-z)) # write your sigmoid function 
    elif function_type == "relu":
        return np.maximum(0, z) # write your relu function
    elif function_type == "tanh":
        return np.tanh(z) # write your tanh function
    else:
        raise ValueError("Unsupported activation function.")



Test your implementation by plotting the functions in the next code cell

In [ ]:

# plot sigmoid and relu
z = np.linspace(-10, 10, 400)
sigmoid_values = activation_function(z, function_type="sigmoid")
relu_values = activation_function(z, function_type="relu")
tanh_values = activation_function(z, function_type="tanh")
fig, ax = plt.subplots(1, 3, figsize=(6, 2))
ax[0].plot(z, sigmoid_values)
ax[0].set_title("Sigmoid")
ax[1].plot(z, relu_values)
ax[1].set_title("ReLU")
ax[2].plot(z, tanh_values)
ax[2].set_title("Tanh")

plt.tight_layout()


Learning through a neuron network, simply amounts to finding a combination of activation functions which best approximates the dataset it is trained on. As we will see later on in this module, this method is powerful as we can flexibly approximate any function using a combination of non-linear functions. 


To get an intuitive understanding of how neurons "learn", we will first see what happens when we change the parameters affecting the activation functions. In the following code cell, you can control two parameters: the slope and the activation point. By changing these two parameters, you can see how the shape of the activation function changes. You will also control the "input" parameter which is the value of x that is fed into the activation function. On the right, you can see how the parameters you change gets reflected in the behaviour of a neuron. 

In [ ]:
# Add an interact to visualise effect of parameters on the neuron output
# Use ReLU, PReLU, sigmoid, and tanh as activation functions
import ipywidgets as widgets
from IPython.display import display
from src.utils import update_neuron_output_using_activation_point
    
slope_slider = widgets.FloatSlider(min=-5, max=5, step=0.1, value=1, description='Slope')
activation_point_slider = widgets.FloatSlider(min=-10, max=10, step=0.1, value=0, description='Act. point')
x_slider = widgets.FloatSlider(min=-10, max=10, step=0.1, value=5, description='Input x')
activation_function_dropdown = widgets.Dropdown(options=['relu', 'sigmoid', 'tanh'], value='relu', description='Activation')
widgets.interact(\
    update_neuron_output_using_activation_point, \
    function_type=activation_function_dropdown, \
    slope=slope_slider, \
    activation_point=activation_point_slider,\
    x=x_slider
);



Next, you will use these sliders to control the shape of three "neurons". What's more, you will also change the weight given to each of these neurons to see the final combined output. Note that the weight can be negative or positive. 

Can you find a set of parameters for these three neurons to approximate the function that you plotted in the beginning of this module?

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from src.utils import update_layer_of_neurons_using_slope_activation_point
min_val, max_val, step = -5, 10, 0.01
weight_min, weight_max = -5, 10

# Sliders
slope_1 = widgets.FloatSlider(min=min_val, max=max_val, step=step, value=0, description='Slope 1')
slope_2 = widgets.FloatSlider(min=min_val, max=max_val, step=step, value=0, description='Slope 2')
slope_3 = widgets.FloatSlider(min=min_val, max=max_val, step=step, value=0, description='Slope 3')

ap_1 = widgets.FloatSlider(min=min_val, max=max_val, step=step, value=0, description='Act. Pt 1')
ap_2 = widgets.FloatSlider(min=min_val, max=max_val, step=step, value=0, description='Act. Pt 2')
ap_3 = widgets.FloatSlider(min=min_val, max=max_val, step=step, value=0, description='Act. Pt 3')

weight_1 = widgets.FloatSlider(min=weight_min, max=weight_max, step=step, value=1, description='Weight 1')
weight_2 = widgets.FloatSlider(min=weight_min, max=weight_max, step=step, value=1, description='Weight 2')
weight_3 = widgets.FloatSlider(min=weight_min, max=weight_max, step=step, value=1, description='Weight 3')

output_bias = widgets.FloatSlider(min=min_val, max=max_val, step=step, value=0, description='Output Bias', 
                                      orientation='vertical')
function_dropdown = widgets.Dropdown(options=['relu', 'sigmoid', 'tanh'], value='relu', description='Activation')

sample_data = (x_sample, y_sample, y_true)
# interactive_output wires up callbacks but does NOT render the widgets itself
out = widgets.interactive_output(
    update_layer_of_neurons_using_slope_activation_point,
    {
        'slope_1': slope_1, 'slope_2': slope_2, 'slope_3': slope_3,
        'activation_point_1': ap_1, 'activation_point_2': ap_2, 'activation_point_3': ap_3,
        'weight_1': weight_1, 'weight_2': weight_2, 'weight_3': weight_3,
        'output_bias': output_bias, "function_type":function_dropdown,
        'sample_data': widgets.fixed(sample_data)
    }
)

# Group sliders by neuron so each column corresponds to one neuron
neuron_1_controls = widgets.VBox([slope_1, ap_1, weight_1])
neuron_2_controls = widgets.VBox([slope_2, ap_2, weight_2])
neuron_3_controls = widgets.VBox([slope_3, ap_3, weight_3])
all_controls = widgets.HBox([
    widgets.VBox([neuron_1_controls, neuron_2_controls, neuron_3_controls, function_dropdown]),
    output_bias
])

# Sliders on the left, plot on the right
display(widgets.HBox([all_controls, out]))

Could you get a good fit? Could you get the error to be less than $10^{-2}$? or even less than $10^{-3}$?

## Think! 
- Why do you think three neurons were sufficient for this curve? 
- What would happen if you had only two neurons? What if you have a million neurons?


# Artificial Neural Networks

In biological neurons, the connection strengths between the cells are modulated by complex biochemical processes. In artificial neurons, the connection strength is simply a number, which we call a weight. The weight can be positive or negative, and it determines how much influence the input has on the output of the neuron. Each neuron performs a linear combination of the inputs with the weights and then applies the non-linear activation function before sending the output to the next layer of neurons. Mathematically, the output of a single neuron can be expressed as:

\begin{equation}
\tag{1}
y = f\left(\sum_{i=1}^{n} w_i x_i + b\right)
\end{equation}

Where:
- $x_i$ are the inputs to the neuron
- $w_i$ are the weights associated with each input, similar to the slope that you just played with
- $b$ is the bias term, which shifts the activation point of the function along with the weights
- $f$ is the activation function (e.g., ReLU, sigmoid)
- $y$ is the output of the neuron

To understand how artificial neuron networks can "learn" to approximate data, you will first build a simple feedforward neural network with one hidden layer. The hidden layer will consist of three neurons, and the output layer will consist of one neuron that produces the final output. So, it is quite similar to the setup you just played with, but now we will have a more formal structure to it.

In particular, we will first write code to do a "forward pass". Here, you take the input data and pass it through the network to get the output. We will then implement "backpropagation" to adjust the parameters of the network based on the error between the predicted output and the true output. 

### Forward pass

In the first task you will write a function to imitate the output of a neuron as shown in Equation 1. This function will take as input the data, the weights, the bias and the activation function, and will return the output of the neuron.

In [ ]:
def neuron(x, weight, bias, function_type="relu"):
    z = np.dot(x, weight) + bias # compute the linear transformation
    y = activation_function(z, function_type) # apply the activation function to z
    return y

In [ ]:
x = x_sample[0] # Take the first sample input for demonstration
y = y_sample[0] # Take the corresponding true output for demonstration
weights = np.array([0.5])
bias = 0.6
output = neuron(x, weights, bias, function_type="relu")
print("Input to the neuron:", x)
print("Output of the neuron:", output)
print("True output:", y)

Great! Now suppose that we have a layer of three neurons, each with its own set of weights and biases. What would be the output of this layer? 


In [ ]:
def layer_of_neurons(x, list_of_neurons, function_type="relu"):
    outputs = []
    for neuron_params in list_of_neurons:
        weight, bias = neuron_params
        output = neuron(x, weight, bias, function_type) # write your function here
        outputs.append(output)
    return np.array(outputs).squeeze() # return the outputs as a numpy array


Let us define three "neurons" which have the same activation function but different weights and biases. We will then combine the output of these three neurons to get the final output.

Each neuron is essentially a list in python with following format
neuron = [weight_vector, bias]
Where:
- weight_vector is a numpy array of shape (n_inputs,) containing the weights for each input
- bias is a numpy array of shape (1,) containing the bias term for the neuron


In [ ]:
w_1_1, b_1_1 = 0.5, 0.6 # weight, bias from input to neuron 1 in layer 1
w_2_1, b_2_1 = -0.5, -0.2 # weight, bias from input to neuron 2 in layer 1
w_3_1, b_3_1 = 0.8, 0.1 # weight, bias from input to neuron 3 in layer 1

neuron_1 = (w_1_1, b_1_1)
neuron_2 = (w_2_1, b_2_1)
neuron_3 = (w_3_1, b_3_1)

In [ ]:
hidden_layer_output = layer_of_neurons(x, [neuron_1, neuron_2, neuron_3], function_type="relu")
print(f"Input to the layer: {x}")
print(f"Output of the layer: {hidden_layer_output}")

Does the shape of the output of the hidden layer make sense to you?

Now, combine the outputs of the hidden layer to get the final output. The final output is a linear combination of the outputs of the hidden layer neurons, weighted by the weights of the output neuron. Usually the output neuron does not have an activation function, so it is simply kept "linear"

In [ ]:
w_1_2, w_2_2, w_3_2 = -0.1, -0.7, 0.5 # weights from hidden layer to output neuron
b_1_2 = 0.2 # bias for output neuron 

output_neuron = (np.array([w_1_2, w_2_2, w_3_2]), b_1_2)

final_output = layer_of_neurons(hidden_layer_output, [output_neuron], function_type="relu")

print(f"Output of the network: {final_output}")
print(f"True output: {y}")


How did the output of the network compare to the true function? Is it close? Is it far? In the next section you will implement a method to quantify the error. 

The error between the predicted output and the true output can be quantified using a loss function. A common loss function for regression problems is the Mean Squared Error (MSE), which is defined as:
\begin{equation}
\tag{2}
MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
\end{equation}

Where:
- $y_i$ is the true output for the $i$-th data point
- $\hat{y}_i$ is the predicted output for the $i$-th data point
- $n$ is the number of data points. In our cause it is just one, since we are only looking at one point at a time, but in general it can be more.


In [ ]:
def convert_scalar_to_array(*value):
    converted_values = []
    for val in value:
        if np.isscalar(val):
            converted_values.append(np.array([val]))
        else:
            converted_values.append(val)
    return converted_values

def mse_loss(y_true, y_pred):
    # check if y_true or y_pred are scalars
    y_true, y_pred = convert_scalar_to_array(y_true, y_pred)
    # write your code to compute mse loss using Equation 2
    n = len(y_true) # number of samples
    loss_value = (1/n) * np.sum((y_true - y_pred) ** 2) 
    
    return loss_value

In [ ]:
mse_loss_value = mse_loss(y, final_output)
print(f"MSE Loss: {mse_loss_value}")

# Backpropagation

Now that we computed the loss between the predicted output and the true output, we need to learn how that is used to adjust the parameters. This is done through a process called backpropagation. Here, the error signal is first computed with respect to the final output and this signal "propagates" back through the network. So the typical information flow through a network is as follows:
- Forward pass: input data flows through the network to produce an output
- Backward pass: the error signal is propagated back through the network to update parameters

First, let us find out how to compute the error signal with respect to the output. This is just the derivative of the loss function with respect to the predicted output. For the MSE loss function, with n=1, this can be computed as follows:


\begin{equation*}
MSE = (y_i - \hat{y}_i)^2
\end{equation*}


\begin{equation}
\tag{3}
\frac{\partial MSE}{\partial \hat{y}_{final}} = \delta_{out} = -2(y_i - \hat{y}_i) 
\end{equation}

The term $\delta_{out}$ is called the output error signal, and it tells us how much the final output of the network is contributing to the error.

In [ ]:
# Compute the loss 
def mse_gradient_output_neuron(y_true, y_pred):
    # check if y_true or y_pred are scalars
    y_true, y_pred = convert_scalar_to_array(y_true, y_pred)

    n = len(y_true) # number of samples
    gradient = (-2/n) * (y_true - y_pred) # write your MSE gradient function here
    return gradient.squeeze() 

In [ ]:
output_error_signal = mse_gradient_output_neuron(y, final_output)
print(f"MSE Gradient: {output_error_signal}")

What is the gradient actually telling us? To understand this let us visualise the loss function and plot the gradient on top of it. 

In [ ]:
from src.utils import plot_loss_landscape_with_state, print_summary_of_network

In [ ]:
states = {
    'Current prediction' : {
        'prediction': final_output,
        'loss': mse_loss_value,
        'condition': 'Current prediction',
        'output_error_signal' : output_error_signal
    },
}

plot_loss_landscape_with_state(
    loss_fn=mse_loss, output_vector=y,
    states=states, tangent=states['Current prediction'], window_size=5
)

What the output error signal is telling us is how steep is the loss function at the current predicted point. This information is required to know what direction to move the parameters in order to reduce the error. Each parameter needs to be updated so that the final prediction moves towards the minima in this loss landscape. This process of updating the parameters to reduce the error is called gradient descent. 

The update rule of gradient descent is given by:

\begin{equation}
\begin{aligned}
\tag{4}
w_i &\leftarrow w_i - \eta \cdot \frac{\partial MSE}{\partial w_i} \\
b_i &\leftarrow b - \eta \cdot \frac{\partial MSE}{\partial b_i} \\
\end{aligned}
\end{equation}

Where $\eta$ is the learning rate, which determines how much we update the parameters in each iteration. Here $w_i$, and $b_i$ are the weights and biases of every neuron in the network.  $\frac{\partial MSE}{\partial w_i}$ and $\frac{\partial MSE}{\partial b_j}$ are the gradients of the loss function with respect to the weights and biases, respectively. Now, write a function to update the weights and biases of the output neuron using the gradients and the learning rate using Equation 4.



In [ ]:
def update_weights(neuron, dL_dw, dL_db, learning_rate=0.01):
    weight, bias = neuron
    weight_update = learning_rate * dL_dw # compute the weight update using the learning rate and the gradient
    bias_update = learning_rate * dL_db # compute the bias update using the learning rate and the gradient
    new_weight = weight - weight_update # update the weight by subtracting the update from the current weight
    new_bias = bias - bias_update # update the bias by subtracting the update from the current bias
    return new_weight, new_bias




We need to  compute these gradients from the output error signal $\delta_{out}$ that we computed above. 

We recursively apply chain rule of calculus to compute the gradients for each layer of the network. Let us first start with the output layer. 

The forward pass of the output neuron is 

\begin{equation}
\tag{5a}
\hat{y}_{final} = w_1^{(2)} y_1^{(1)} + w_2^{(2)} y_2^{(1)} + w_3^{(2)} y_3^{(1)} + b_1^{(2)}
\end{equation}

Where $y_i^{(1)}$, $\forall$ $i = \{1, 2, 3\}$ are the outputs of the three neurons in the first layer, and $w_i^{(2)}$ are the weights of the output neuron. Note that the output neuron does not have an activation function, so the output is simply a linear combination of the outputs of the hidden layer neurons. Later we will see how to compute the gradients for the hidden layer neurons, which do have an activation function.

\begin{equation*}
\frac{\partial \hat{y}_{final}}{\partial w_1^{(2)}} = y_1^{(1)}
\end{equation*}

\begin{equation*}
\frac{\partial \hat{y}_{final}}{\partial w_1^{(2)}} = y_1^{(1)}
\end{equation*}

\begin{equation*}
\frac{\partial \hat{y}_{final}}{\partial w_3^{(2)}} = y_3^{(1)}
\end{equation*}

\begin{equation*}
\frac{\partial \hat{y}_{final}}{\partial b_1^{(2)}} = 1
\end{equation*}


In [ ]:
# Plug in the right variables to get the correct gradients. For instance, 
y_final = final_output # final output of the network is called y_final

y_1_1 = hidden_layer_output[0] # what should go here?
y_2_1 = hidden_layer_output[1] # what should go here?
y_3_1 = hidden_layer_output[2] # what should go here?

dy_final_dw_1_2 = y_1_1 # write the correct variable here
dy_final_dw_2_2 = y_2_1 # write the correct variable here
dy_final_dw_3_2 = y_3_1 # write the correct variable here
dy_final_db_1_2 = 1 # the derivative of the output with respect to the bias is always 1

Now, we can use the chain rule to calculate the gradient of the loss function $MSE$ with respect to the weights and bias of the output neuron. For example, for weight $w_1^{(2)}$, we have:


\begin{equation}
\begin{aligned}
\tag{6}
\frac{\partial MSE}{\partial w_1^{(2)}} &= \frac{\partial MSE}{\partial \hat{y}_{final}} \cdot \frac{\partial \hat{y}_{final}}{\partial w_1^{(2)}} = \delta_{out} \cdot y_1^{(1)} \\
\frac{\partial MSE}{\partial w_2^{(2)}} &= \frac{\partial MSE}{\partial \hat{y}_{final}} \cdot \frac{\partial \hat{y}_{final}}{\partial w_2^{(2)}} = \delta_{out} \cdot y_2^{(1)} \\
\frac{\partial MSE}{\partial w_3^{(2)}} &= \frac{\partial MSE}{\partial \hat{y}_{final}} \cdot \frac{\partial \hat{y}_{final}}{\partial w_3^{(2)}} = \delta_{out} \cdot y_3^{(1)} \\
\frac{\partial MSE}{\partial b_1^{(2)}} &= \frac{\partial MSE}{\partial \hat{y}_{final}} \cdot \frac{\partial \hat{y}_{final}}{\partial b_1^{(2)}} = \delta_{out}
\end{aligned}
\end{equation}

Now, write the code to calculate the gradients for the weights and bias of the output neuron using Equation 6.



In [ ]:
# write the correct expression for the gradient of the loss with respect to weight 1 in layer 2
dL_dw_1_2 = output_error_signal * dy_final_dw_1_2 
# write the correct expression for the gradient of the loss with respect to weight 2 in layer 2
dL_dw_2_2 = output_error_signal * dy_final_dw_2_2 
# write the correct expression for the gradient of the loss with respect to weight 3 in layer 2
dL_dw_3_2 = output_error_signal * dy_final_dw_3_2
# write the correct expression for the gradient of the loss with respect to bias in layer 2
dL_db_1_2 = output_error_signal * dy_final_db_1_2

Now, update the weights and bias of the output neuron using the gradients that we calculated. After updating, calculate the new loss and see if it has decreased.

In [ ]:
# update the weights and bias for neuron 1 in layer 2
learning_rate = 0.001 
updated_output_neuron = update_weights(\
    neuron=output_neuron, \
    dL_dw=np.array([dL_dw_1_2, dL_dw_2_2, dL_dw_3_2]), \
    dL_db=dL_db_1_2, \
    learning_rate=learning_rate, \
)

print(f"Updated output neuron weights: {updated_output_neuron[0]}, bias: {updated_output_neuron[1]}")
print(f"Original output neuron weights: {output_neuron[0]}, bias: {output_neuron[1]}")

Now let us also calculate the gradients for the weights and biases of the three neurons in the previous layer. The forward pass for each of these neurons is given by:
\begin{equation*}
\begin{aligned}
y_1^{(1)} &= f(w_1{^{(1)}} x + b_1{^{(1)}}) = f(z_1^{(1)}) \\
y_2^{(1)} &= f(w_2{^{(1)}} x + b_2{^{(1)}}) = f(z_2^{(1)}) \\
y_3^{(1)} &= f(w_3{^{(1)}} x + b_3{^{(1)}}) = f(z_3^{(1)})
\end{aligned}
\end{equation*}

Where $f$ is the activation function (ReLU in our case). We will only show the chain rule for the first neuron to keep the text simple to read, but the same process applies to the other two neurons as well. 

Derivative of the $y_1^{(1)}$ with respect to its parameters is given by:
\begin{equation*}
\begin{aligned}
\frac{\partial y_1^{(1)}}{\partial w_1^{(1)}} &= \frac{\partial y_1^{(1)}}{\partial z_1^{(1)}} \cdot \frac{\partial z_1^{(1)}}{\partial w_1{^{(1)}}} \\
\frac{\partial y_1^{(1)}}{\partial b_{1^{(1)}}} &= \frac{\partial y_1^{(1)}}{\partial z_1^{(1)}} \cdot \frac{\partial z_1^{(1)}}{\partial b_1{^{(1)}}} 
\end{aligned}
\end{equation*}

Now, we need to calculate the derivative of the activation function with respect to its input $z_1^{(1)}$. The derivative of the activation function can be represented by the symbol $f'$. 
For ReLU, the derivative is given by:
(Note that derivative for ReLU is not defined at 0, but we can simply set it to 0 by convention, since it does not affect the learning process much)

For neuron $i$ at layer $j$, we can simply write the derivative of the activation function as $f'^{(j)}_i$.

\begin{equation*}
f'^{(1)}(z_i) = \frac{\partial y_i^{(j)}}{\partial z_i^{(j)}} = 
\begin{cases}
1 & \text{if } z_i > 0 \\
0 & \text{if } z_i \leq 0
\end{cases}
\end{equation*}


This gives, 
\begin{equation*}
\begin{aligned}
\frac{\partial y_1^{(1)}}{\partial w_{1^{(1)}}} &= \frac{\partial y_1^{(1)}}{\partial z_1^{(1)}} \cdot \frac{\partial z_1^{(1)}}{\partial w_{1^{(1)}}} = f'_1 \cdot x \\
\frac{\partial y_1^{(1)}}{\partial b_{1^{(1)}}} &= \frac{\partial y_1^{(1)}}{\partial z_1^{(1)}} \cdot \frac{\partial z_1^{(1)}}{\partial b_{1^{(1)}}} = f'_1 \cdot 1
\end{aligned}
\end{equation*}

What is the derivative of the output value $\hat{y}_{final}$ with respect to the output of the first neuron $y_1^{(1)}$? Refer to the forward pass of the output neuron in Equation 5 to calculate this derivative.

\begin{equation*}
\frac{\partial \hat{y}_{final}}{\partial y_1^{(1)}} = w_1^{(2)}
\end{equation*}
Now, we can use the chain rule to calculate the gradient of the loss function with respect to the weights and bias of the first neuron. For example, for weight $w_1^{(1)}$, we have:

\begin{equation}
\begin{aligned}
\tag{7a}
\frac{\partial MSE}{\partial w_1^{(1)}} &= \frac{\partial MSE}{\partial \hat{y}_{final}} \cdot \frac{\partial \hat{y}_{final}}{\partial y_1^{(1)}} \cdot \frac{\partial y_1^{(1)}}{\partial w_{1^{(1)}}} \\
&= \delta_{out} \cdot w_1^{(2)} \cdot f'_1 \cdot x
\end{aligned}
\end{equation}

and for bias $b_1^{{(1)}}$, we have:

\begin{equation}
\begin{aligned}
\tag{7b}
\frac{\partial MSE}{\partial b_1^{{(1)}}} &= \frac{\partial MSE}{\partial \hat{y}_{final}} \cdot \frac{\partial \hat{y}_{final}}{\partial y_1^{(1)}} \cdot \frac{\partial y_1^{(1)}}{\partial b_{1^{(1)}}} \\
&= \delta_{out} \cdot w_1^{(2)} \cdot f'_1 
\end{aligned}
\end{equation}
Now, write the code to calculate the gradients for the weights and bias of the first neuron using Equations 7a and 7b. You can repeat the same process to calculate the gradients for the other two neurons in the hidden layer as well.


In [ ]:
from src.utils import relu_derivative, sigmoid_derivative, tanh_derivative

In [ ]:
# write the correct expression for the gradient of the loss with respect to weight 1 in layer 1
dL_dw_1_1 = output_error_signal * w_1_2 * relu_derivative(y_1_1) * x 
dL_db_1_1 = output_error_signal * w_1_2 * relu_derivative(y_1_1) * 1
print(f"Gradient for weight 1 in layer 1: {dL_dw_1_1}, Gradient for bias in layer 1: {dL_db_1_1}")


In [ ]:
# write the correct expression for the gradient of the loss with respect to weight 2 in layer 1
dL_dw_2_1 = output_error_signal * w_2_2 * relu_derivative(y_2_1) * x
dL_db_2_1 = output_error_signal * w_2_2 * relu_derivative(y_2_1) * 1

In [ ]:
# write the correct expression for the gradient of the loss with respect to weight 3 in layer 1
dL_dw_3_1 = output_error_signal * w_3_2 * relu_derivative(y_3_1) * x
dL_db_3_1 = output_error_signal * w_3_2 * relu_derivative(y_3_1) * 1

In [ ]:
# update the weights and bias for neuron 1 in layer 1
updated_neuron_1 = update_weights(\
    neuron=neuron_1, \
    dL_dw=dL_dw_1_1, \
    dL_db=dL_db_1_1, \
    learning_rate=learning_rate, \
)
# update the weights and bias for neuron 2 in layer 1
updated_neuron_2 = update_weights(\
    neuron=neuron_2, \
    dL_dw=dL_dw_2_1, \
    dL_db=dL_db_2_1, \
    learning_rate=learning_rate, \
)
# update the weights and bias for neuron 3 in layer 1
updated_neuron_3 = update_weights(\
    neuron=neuron_3, \
    dL_dw=dL_dw_3_1, \
    dL_db=dL_db_3_1, \
    learning_rate=learning_rate, \
)


In [ ]:
# print the updated weights and biases for all neurons in the network
print(f"Updated neuron 1 weight: {updated_neuron_1[0]}, bias: {updated_neuron_1[1]}")
print(f"Updated neuron 2 weight: {updated_neuron_2[0]}, bias: {updated_neuron_2[1]}")
print(f"Updated neuron 3 weight: {updated_neuron_3[0]}, bias: {updated_neuron_3[1]}")


In [ ]:
# What is the new prediction of the network after the weight updates? 
# Create a function called forward_pass that takes the input x
# and two layers of neurons and returns the final output of the network
def forward_pass(x, layer_1_neurons, layer_2_neurons, function_type="relu"):
    hidden_layer_output = layer_of_neurons(x, layer_1_neurons, function_type)
    final_output = layer_of_neurons(hidden_layer_output, layer_2_neurons, function_type)
    return final_output


In [ ]:
updated_layer_1_neurons = [updated_neuron_1, updated_neuron_2, updated_neuron_3]
updated_layer_2_neurons = [updated_output_neuron]
new_prediction = forward_pass(\
    x=x, \
    layer_1_neurons=updated_layer_1_neurons, \
    layer_2_neurons=updated_layer_2_neurons, \
    function_type="relu"
)
print(f"New prediction after weight updates: {new_prediction}")
print(f"Earlier prediction before weight updates: {final_output}")
print(f"True output: {y}")

Is it going in the right direction? Run the next code cell to visualise the changes

In [ ]:
updated_loss = mse_loss(y, new_prediction)
updated_output_error_signal = mse_gradient_output_neuron(y, new_prediction)

states.update({
    'Updated prediction' : {
        'prediction': new_prediction,
        'loss': updated_loss,
        'condition': 'After one weight update step',
        'output_error_signal' : updated_output_error_signal
    },
})
plot_loss_landscape_with_state(
    loss_fn=mse_loss, output_vector=y,
    states=states, tangent=states['Updated prediction'], window_size=5
)

Now, write a loop which repeats this process of forward pass multiple times. In each iteration, calculate the loss and update the parameters using backpropagation. You should see the loss decreasing over iterations as the network learns to approximate the function.



In [ ]:
n_iter = 50
x_train = x_sample[0] # again we will use the first sample input for training
y_train = y_sample[0] # corresponding true output for training
learning_rate = 0.1
states_training = {}
for i in range(n_iter):
    # Forward pass
    if i == 0:
        # initialise random weights and biases for the first iteration
        w_1_1, b_1_1 = 0.5, 0.6 # weight, bias from input to neuron 1 in layer 1
        w_2_1, b_2_1 = -0.5, -0.2
        w_3_1, b_3_1 = 0.8, 0.1
        w_1_2, w_2_2, w_3_2 = -0.1, -0.7, 0.5
        b_1_2 = 0.2

    neuron_1 = (w_1_1, b_1_1)
    neuron_2 = (w_2_1, b_2_1)
    neuron_3 = (w_3_1, b_3_1)
    output_neuron = (np.array([w_1_2, w_2_2, w_3_2]), b_1_2)
    layer_1_neurons = [neuron_1, neuron_2, neuron_3]
    layer_2_neurons = [output_neuron]
    hidden_layer_1 = layer_of_neurons(x_train, layer_1_neurons, function_type="relu")
    final_output = layer_of_neurons(hidden_layer_1, layer_2_neurons, function_type="relu")
    
    loss_value = mse_loss(y_train, final_output)
    output_error_signal = mse_gradient_output_neuron(y_train, final_output)

    # Compute gradients for output layer
    dL_dw_1_2 = output_error_signal * hidden_layer_1[0]
    dL_dw_2_2 = output_error_signal * hidden_layer_1[1]
    dL_dw_3_2 = output_error_signal * hidden_layer_1[2]
    dL_db_1_2 = output_error_signal * 1

    # Compute gradients for hidden layer
    dL_dw_1_1 = output_error_signal * w_1_2 * relu_derivative(hidden_layer_1[0]) * x_train
    dL_db_1_1 = output_error_signal * w_1_2 * relu_derivative(hidden_layer_1[0]) * 1
    dL_dw_2_1 = output_error_signal * w_2_2 * relu_derivative(hidden_layer_1[1]) * x_train
    dL_db_2_1 = output_error_signal * w_2_2 * relu_derivative(hidden_layer_1[1]) * 1
    dL_dw_3_1 = output_error_signal * w_3_2 * relu_derivative(hidden_layer_1[2]) * x_train
    dL_db_3_1 = output_error_signal * w_3_2 * relu_derivative(hidden_layer_1[2]) * 1

    # Update weights and biases for output layer
    output_neuron = update_weights(\
        neuron=output_neuron, \
        dL_dw=np.array([dL_dw_1_2, dL_dw_2_2, dL_dw_3_2]), \
        dL_db=dL_db_1_2, \
        learning_rate=learning_rate, \
    )

    # Update weights and biases for hidden layer
    neuron_1 = update_weights(\
        neuron=neuron_1, \
        dL_dw=dL_dw_1_1, \
        dL_db=dL_db_1_1, \
        learning_rate=learning_rate, \
    )
    neuron_2 = update_weights(\
        neuron=neuron_2, \
        dL_dw=dL_dw_2_1, \
        dL_db=dL_db_2_1, \
        learning_rate=learning_rate, \
    )
    neuron_3 = update_weights(\
        neuron=neuron_3, \
        dL_dw=dL_dw_3_1, \
        dL_db=dL_db_3_1, \
        learning_rate=learning_rate, \
    )

    w_1_1, b_1_1 = neuron_1
    w_2_1, b_2_1 = neuron_2
    w_3_1, b_3_1 = neuron_3
    w_1_2, w_2_2, w_3_2 = output_neuron[0]
    b_1_2 = output_neuron[1]
    
    
    states_training["iteration_{}".format(i+1)] = {
        'prediction': final_output,
        'loss': loss_value,
        'condition': f'Iteration {i+1}',
        'output_error_signal' : output_error_signal
    }


In [ ]:
plot_loss_landscape_with_state(
    loss_fn=mse_loss, output_vector=y_train,
    states=states_training, show_legend=False, window_size=5, limit_y_axis=False,
)

In [ ]:
num_iterations = len(states_training)
losses = [state['loss'] for state in states_training.values()]
plt.figure(figsize=(5, 3))
plt.plot(range(1, num_iterations + 1), losses, marker='o')
plt.title('Loss History Over Iterations')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.yscale('log')

    

Did the loss decrease? How close did the final output get to the true function?

## Training the neural network using PyTorch

The code that we have written so far is a very basic implementation of a neural network. It is not efficient and it does not take advantage of the powerful libraries that are available for deep learning. One such library is PyTorch, which provides a high-level interface for building and training neural networks. Let us use that to build a similar neural network and train it to learn the function that we generated at the beginning of this module.

In [ ]:
import torch
import torch.nn as nn

In [ ]:
model = nn.Sequential(
    nn.Linear(1, 3), 
    nn.ReLU(), 
    nn.Linear(3, 1)
)


In [ ]:
# print model weights and biases 
for name, param in model.named_parameters():
    if param.requires_grad:
        # name, value, activation function (if applicable)
        print(f"{name}: {param.data}")
        

Call the PyTorch functions to set the optimiser and the loss function

In [ ]:
learning_rate_torch = 0.1
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate_torch)

We now need to set up a data preprocessing pipeline. This is essential because we would like the network to see data in a standardised scale. The standard preprocessing is to subtract the mean and divide by the standard deviation of the data. This is called standardisation.


In [ ]:
def standardise_data(x):
    mean = np.mean(x)
    std = np.std(x)
    standard_data = (x - mean) / std
    return standard_data, mean, std

In [ ]:
# train the model on the same data for comparison
x_train_standard, mean, std = standardise_data(x_sample)
y_train_standard, mean_y, std_y = standardise_data(y_sample)

In [ ]:
# Convert to PyTorch tensors
x_train_tensor = torch.tensor(x_train_standard, dtype=torch.float32).unsqueeze(1) # shape (n_samples, 1)
y_train_tensor = torch.tensor(y_train_standard, dtype=torch.float32).unsqueeze(1) # shape (n_samples, 1)


In [ ]:
# Start training runs: 
num_epochs = 1000
loss_history_torch = []
for epoch in range(num_epochs):
    # Set the model to training mode. 
    model.train()
    # Zero the gradients before running the backward pass. 
    optimizer.zero_grad()
    # Forward pass: compute the model output and loss
    outputs = model(x_train_tensor)
    # Compute the loss between the model output and the true labels
    loss = criterion(outputs, y_train_tensor)
    # Backward pass: compute the gradients of the loss with respect to the model parameters
    loss.backward()
    # Update the model parameters using the optimizer
    optimizer.step()
    # Append the current loss to the loss history list 
    loss_history_torch.append(loss.item())

In [ ]:
loss_history_torch = np.array(loss_history_torch)
plt.plot(loss_history_torch)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.yscale("log")

In [ ]:
# Plot the predictions vs true values after training
def predict(model, x, mean_x, std_x, mean_y, std_y):
    # Set the model to evaluation mode. 
    model.eval()
    with torch.no_grad(): # Make predictions without tracking gradients
        # Standardise the input data using the mean and std from training
        x_standard = (x - mean_x) / std_x 
        # Convert the standardised input to a PyTorch tensor 
        x_tensor = torch.tensor(x_standard, dtype=torch.float32).unsqueeze(1)
        # Get the model predictions for the input data and convert to numpy array
        y_pred_standard = model(x_tensor).squeeze().numpy()
        # Unstandardise the predictions using the mean and std from training
        y_pred = y_pred_standard * std_y + mean_y
    return y_pred




In [ ]:
# Let us see how the model performs on the true x values after training
y_pred = predict(model, x_true, mean, std, mean_y, std_y)


In [ ]:
# Plot the predictions vs true values after training
plt.plot(x_true, y_true, color="black", label="True Function", linestyle='--', alpha=0.2)
plt.plot(x_true, y_pred, color="green", label="PyTorch Model Prediction")
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()

Now, increase the number of parameters in the PyTorch model to see how it fits the data. 

In [ ]:
num_params = 64
model_complex = nn.Sequential(
    nn.Linear(1, num_params),
    nn.ReLU(),
    nn.Linear(num_params, 1)
)



In [ ]:
def train_model(model, x_train, y_train, epochs, lr=0.001):
    # define the loss function and optimizer
    criterion = nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_history = []
    for epoch in range(epochs):
        # Set the model to training mode.
        model.train()
        # Zero the gradients before running the backward pass.
        optimizer.zero_grad()
        # Forward pass: compute the model output and loss
        outputs = model(x_train)
        # Compute the loss between the model output and the true labels
        loss = criterion(outputs, y_train)
        # Backward pass: compute the gradients of the loss with respect to the model parameters
        loss.backward()
        # Update the model parameters using the optimizer
        optimizer.step()
        # Append the current loss to the loss history list
        loss_history.append(loss.item())
        
    return model, np.array(loss_history)

In [ ]:
model_complex_trained, loss_history_complex = train_model(\
    model_complex, x_train_tensor, y_train_tensor, epochs=1000, lr=0.01
)

In [ ]:
# Plot the loss history for the complex model
plt.plot(loss_history_complex)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.yscale("log")

In [ ]:
x_test = np.linspace(0, 10, 100)
y_test = degree_3_polynomial(x_test, coeffs=[A, B, C, D], noise_std=0.2)
y_predictions_complex = predict(model_complex_trained, x_test, mean, std, mean_y, std_y)
plt.plot(x_test, y_test, color="black", label="True Function", linestyle='--', alpha=0.2)
plt.plot(x_test, y_predictions_complex, 'm-', label='Predictions (Complex PyTorch Model)')
plt.legend()
plt.xlabel('X')
plt.ylabel('Y')
plt.ylim(-5, 15)


# Universal approximation theorem

- Increasing the number of neurons allows the network to learn more features. In case of ReLu, this is done by fitting piecewise linear functions to the data. 
- Increasing depth allows exponential increase in representational power. For example, a network with one hidden layer can approximate any function, but it might require a very large number of neurons. On the other hand, a deeper network can approximate the same function with fewer neurons by learning hierarchical features.

\begin{equation}
N_{representation} \propto N_{neurons}^{L}
\end{equation}

Where $N_{representation}$ is the number of functions that can be represented by the network, $N_{neurons}$ is the number of neurons in each layer, and $L$ is the number of layers.

However, increasing depth also makes the network harder to train. As gradients are propagated backward through the network, they can become very small because of the chain rule. If the gradients become too small, the weights of the earliest layers will not be updated effectively, and the network will not learn. This is known as the vanishing gradient problem.

You can also have exploding gradients if you set the learning rate too high. This can result in large jumps in the loss landscape and can cause the training to diverge. 

# Comparing with polynomial regression


In [ ]:
polynomial_degree = 20
num_neurons = 200 

# fit a polynomial for the same data for comparison
polynomial_coefficients = np.polyfit(x_sample, y_sample, polynomial_degree)
y_predictions_polynomial = np.polyval(polynomial_coefficients, x_test)

# fit a neural network with more neurons for the same data for comparison
model_more_neurons = nn.Sequential(
    nn.Linear(1, num_neurons),
    nn.ReLU(),
    nn.Linear(num_neurons, 1)
)
model_more_neurons_trained, loss_history_more_neurons = train_model(
    model_more_neurons, x_train_tensor, y_train_tensor, epochs=1000, lr=0.01
)
y_predictions_more_neurons = predict(model_more_neurons_trained, x_test, mean, std, mean_y, std_y)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].plot(x_test, y_test, color="black", label="True Function", linestyle='--')
ax[0].scatter(x_sample, y_sample, color="red", label="Sampled Data")
ax[0].plot(x_test, y_predictions_polynomial, 'c-', label=f'Predictions (Polynomial Degree {polynomial_degree})')
ax[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
ax[0].set_ylim(-5, 15)
ax[0].set_xlabel('X')
ax[0].set_ylabel('Y')
ax[1].plot(x_test, y_test, color="black", label="True Function", linestyle='--')
ax[1].scatter(x_sample, y_sample, color="red", label="Sampled Data")
ax[1].plot(x_test, y_predictions_more_neurons, 'y-', label=f'Predictions (NN with {num_neurons} Neurons)')
ax[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
ax[1].set_ylim(-5, 15)
ax[1].set_xlabel('X')
ax[1].set_ylabel('Y')

fig.tight_layout()
